In [1]:
%matplotlib

Using matplotlib backend: module://matplotlib_inline.backend_inline


In [2]:
import mne
from pprint import pprint
import json
import csv
import numpy as np
import pandas as pd
from pathlib import Path
from mne.preprocessing import (ICA, create_eog_epochs, create_ecg_epochs, corrmap)
import mne_bids

In [17]:
# Do it from the .con file
root_data_path = Path("/Users/neuroling/Downloads")
eeg_data_path = root_data_path / "sample_audvis_raw.fif"

print(root_data_path)

LDT_FIFfile = eeg_data_path  #/ Path('de%.3d-2-raw.fif' %sub_id)

#LDT_raw.plot()
#events = mne.find_events(LDT_raw, stim_channel='STI 014') #將集合而成的channel命名為STI 014

LDT_raw = mne.io.read_raw_fif(LDT_FIFfile, preload=True)

print(LDT_raw.info)
eeg_sfrq = LDT_raw.info['sfreq']
print(eeg_sfrq)
LDT_raw.plot()
"""
n_LDT_raw = LDT_raw.copy()

n_LDT_raw.load_data().pick_types(meg=True, stim=True).filter(0, 30, phase='zero-double').resample(500)
print(n_LDT_raw.info)
#print(events.shape)

n_LDT_raw.plot()
"""

/Users/neuroling/Downloads
Opening raw data file /Users/neuroling/Downloads/sample_audvis_raw.fif...
    Read a total of 3 projection items:
        PCA-v1 (1 x 102)  idle
        PCA-v2 (1 x 102)  idle
        PCA-v3 (1 x 102)  idle
    Range : 25800 ... 192599 =     42.956 ...   320.670 secs
Ready.
Reading 0 ... 166799  =      0.000 ...   277.714 secs...
<Info | 21 non-empty values
 acq_pars: ACQch001 110113 ACQch002 110112 ACQch003 110111 ACQch004 110122 ...
 bads: 2 items (MEG 2443, EEG 053)
 ch_names: MEG 0113, MEG 0112, MEG 0111, MEG 0122, MEG 0123, MEG 0121, MEG ...
 chs: 204 Gradiometers, 102 Magnetometers, 9 Stimulus, 60 EEG, 1 EOG
 custom_ref_applied: False
 description: acquisition (megacq) VectorView system at NMR-MGH
 dev_head_t: MEG device -> head transform
 dig: 146 items (3 Cardinal, 4 HPI, 61 EEG, 78 Extra)
 events: 1 item (list)
 experimenter: MEG
 file_id: 4 items (dict)
 highpass: 0.1 Hz
 hpi_meas: 1 item (list)
 hpi_results: 1 item (list)
 lowpass: 172.2 Hz
 meas_d

"\nn_LDT_raw = LDT_raw.copy()\n\nn_LDT_raw.load_data().pick_types(meg=True, stim=True).filter(0, 30, phase='zero-double').resample(500)\nprint(n_LDT_raw.info)\n#print(events.shape)\n\nn_LDT_raw.plot()\n"

Channels marked as bad:
['MEG 2443', 'EEG 053']


In [4]:
events = mne.find_events(LDT_raw)
print(f">> Type of events: {type(events)}\n")
print(len(events))
#print(events)

# create event id and label them according to the task paradigm.
event_id = {
    "Auditory/Left": 1,
    "Auditory/Right": 2,
    "Visual/Left": 3,
    "Visual/Right": 4,
    "Smiley": 5,
    "Button": 32
}

#print(raw.annotations)

Finding events on: STI 014
320 events found on stim channel STI 014
Event IDs: [ 1  2  3  4  5 32]
>> Type of events: <class 'numpy.ndarray'>

320


In [5]:
annotations = mne.annotations_from_events(events,event_desc={v:k for k,v in event_id.items()},sfreq=LDT_raw.info["sfreq"])
print(annotations)

<Annotations | 320 segments: Auditory/Left (72), Auditory/Right (73), ...>


In [9]:
print(LDT_raw.annotations)

<Annotations | 0 segments>


In [12]:
print(27977/eeg_sfrq)

46.58058898776848


In [13]:
# Specify the power line frequency as that is not included in the raw info
# Filter again without dropping bads
import os
from IPython.display import clear_output
import time
#PROJECT_FOLDER = ""
#if os.get_cwd() != PROJECT_FOLDER:  <- make sure you are in the right folder
#    os.chdir(PROJECT_FOLDER) 

raw_cropped_filtered = LDT_raw.copy().filter(l_freq=1, h_freq=40,
                                                 method="fir",
                                                 verbose=True)
if raw_cropped_filtered.info["line_freq"] is None:    
    raw_cropped_filtered.info["line_freq"] = 60

# Set the path to where it will be saved
out_path = Path("out_data/sample_BIDS")

# Set path with data specific variables for naming directory files
bids_path = mne_bids.BIDSPath(subject="01",
                              session="01",
                              task="audiovisual",
                              run="01",
                              root=out_path)
# Write raw to BIDS!
# for some reason though, raw.annotations and events.tsv onset time doesn't match...
mne_bids.write_raw_bids(raw_cropped_filtered, bids_path=bids_path, events=events, event_id=event_id,
                        overwrite=True,format= "FIF", allow_preload=True )

clear_output(wait=True)  # clear verbose output
time.sleep(5)
mne_bids.print_dir_tree(out_path)
print("\n\n")       # print the created file & directory structure
# data summary
display(mne_bids.make_report(out_path, verbose="INFO"))  # verbose: True is the same as 'INFO', False same as 'WARNING'.

|sample_BIDS/
|--- README
|--- dataset_description.json
|--- participants.json
|--- participants.tsv
|--- sub-01/
|------ ses-01/
|--------- sub-01_ses-01_scans.tsv
|--------- meg/
|------------ sub-01_ses-01_coordsystem.json
|------------ sub-01_ses-01_task-audiovisual_run-01_channels.tsv
|------------ sub-01_ses-01_task-audiovisual_run-01_events.json
|------------ sub-01_ses-01_task-audiovisual_run-01_events.tsv
|------------ sub-01_ses-01_task-audiovisual_run-01_meg.fif
|------------ sub-01_ses-01_task-audiovisual_run-01_meg.json



Summarizing participants.tsv out_data/sample_BIDS/participants.tsv...
Summarizing scans.tsv files [PosixPath('out_data/sample_BIDS/sub-01/ses-01/sub-01_ses-01_scans.tsv')]...
The participant template found: sex were all unknown;
handedness were all unknown;
ages all unknown


' The [Unspecified] dataset was created by [Unspecified1], and [Unspecified2] and\nconforms to BIDS version 1.9.0. This report was generated with MNE-BIDS\n(https://doi.org/10.21105/joss.01896). The dataset consists of 1 participants\n(sex were all unknown; handedness were all unknown; ages all unknown) and 1\nrecording sessions: 01. Data was recorded using an MEG system (Elekta) sampled\nat 600.61 Hz with line noise at 60.0 Hz. The following software filters were\napplied during recording: SpatialCompensation. There was 1 scan in total.\nRecording durations ranged from 277.71 to 277.71 seconds (mean = 277.71, std =\n0.0), for a total of 277.71 seconds of data recorded over all scans. For each\ndataset, there were on average 376.0 (std = 0.0) recording channels per scan,\nout of which 374.0 (std = 0.0) were used in analysis (2.0 +/- 0.0 were removed\nfrom analysis).'

In [14]:
## NOT functioning. ## 
import json

EVENTS_TSV_PATH = "/Users/neuroling/Downloads/sub-01/ses-01/meg/sub-01_ses-01_task-audiovisual_run-01_events.tsv"
EVENTS_JSON_PATH = "/Users/neuroling/Downloads/sub-01/ses-01/meg/sub-01_ses-01_task-audiovisual_run-01_events.json"

json_data ={
    "onset": {
        "Description": "Onset (in seconds) of the event from the beginning of the first datapoint. Negative onsets account for events before the first stored data point.",
        "Units": "s"
    },
    "duration": {
        "Description": "Duration of the event in seconds from onset. Must be zero, positive, or 'n/a' if unavailable. A zero value indicates an impulse event. ",
        "Units": "s"
    },
    "sample": {
        "Description": "The event onset time in number of sampling points.First sample is 0."
    },
    "value": {
        "Description": "The event code (also known as trigger code or event ID) associated with the event."
    },
    "trial_type": {
        "Description": "The type, category, or name of the event."
    }
}


try:
    df_events = pd.read_csv(EVENTS_TSV_PATH, sep="\t")

except FileNotFoundError:
    print("... Manually writing our own events.tsv")
    sfreq = raw_cropped_filtered.info["sfreq"]
    events_cropped = mne.find_events(raw_cropped_filtered)
    df_events = df_annot.iloc[:len(events_cropped)]
    df_events = df_events.rename(columns={"description": "trial_type"})
    df_events = df_events.drop(columns=["orig_time", "extras"])
    df_events.insert(3, "value", events_cropped.T[2])
    df_events.insert(4, "sample", events_cropped.T[0])
    df_events["onset"] = df_events["onset"] - raw_cropped_filtered.first_time
    df_events["sample"] = df_events["sample"] - int(round(raw_cropped_filtered.first_time * sfreq))
    df_events.to_csv(EVENTS_TSV_PATH, sep="\t")
    with open(EVENTS_JSON_PATH, "w") as f:
        json.dump(json_data, f)

finally:
    display(df_events)

!ls "out_data/sample_BIDS/sub-01/ses-01/meg"

... Manually writing our own events.tsv
Finding events on: STI 014
320 events found on stim channel STI 014
Event IDs: [ 1  2  3  4  5 32]


NameError: name 'df_events' is not defined